In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import tensorflow_datasets as tfds
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from kerastuner import RandomSearch
from tensorflow.keras.datasets import mnist


In [ ]:
(x_train,y_train),(x_test,y_test) = mnist.load_data()

In [ ]:
base_model = tf.keras.applications.VGG16(
    weights='imagenet',
    input_shape=(224, 224, 3),)

for layer in base_model.layers:
    layer.trainable = False

In [ ]:
model = Sequential([
    base_model,
    Flatten(),
    Dense(128, activation='relu'),
    Dense(5, activation='softmax')
]
)

In [ ]:
model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
dataset,info=tfds.load('tf_flowers',with_info=True,as_supervised=True)

In [ ]:
info

tfds.core.DatasetInfo(
    name='tf_flowers',
    full_name='tf_flowers/3.0.1',
    description="""
    A large set of images of flowers
    """,
    homepage='https://www.tensorflow.org/tutorials/load_data/images',
    data_dir='/root/tensorflow_datasets/tf_flowers/3.0.1',
    file_format=tfrecord,
    download_size=218.21 MiB,
    dataset_size=221.83 MiB,
    features=FeaturesDict({
        'image': Image(shape=(None, None, 3), dtype=uint8),
        'label': ClassLabel(shape=(), dtype=int64, num_classes=5),
    }),
    supervised_keys=('image', 'label'),
    disable_shuffling=False,
    nondeterministic_order=False,
    splits={
        'train': <SplitInfo num_examples=3670, num_shards=2>,
    },
    citation="""@ONLINE {tfflowers,
    author = "The TensorFlow Team",
    title = "Flowers",
    month = "jan",
    year = "2019",
    url = "http://download.tensorflow.org/example_images/flower_photos.tgz" }""",
)

In [ ]:
n_classes=info.features['label'].num_classes
print(n_classes)

5


In [ ]:
IMG_SIZE=224

In [ ]:
images,labels=[],[]
for image,label in tfds.as_numpy(dataset['train']):
    image=tf.image.resize(image,(IMG_SIZE,IMG_SIZE))
    images.append(image)
    labels.append(label)

In [ ]:
type(images)
images=np.array(images)
labels=np.array(labels)

In [ ]:
images.shape

(3670, 224, 224, 3)

In [ ]:
labels.shape

(3670,)

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(images,labels, test_size=0.2,random_state=42)

In [ ]:
train_data_generator=ImageDataGenerator(rescale=1./255)
test_data_generator=ImageDataGenerator(rescale=1./255)

In [ ]:
train_data_generator=train_data_generator.flow(
   tf.image.resize(x_train, (IMG_SIZE, IMG_SIZE)).numpy(),
    tf.keras.utils.to_categorical(y_train, num_classes=n_classes)
)
test_data_generator=test_data_generator.flow(
   tf.image.resize(x_test, (IMG_SIZE, IMG_SIZE)).numpy(),
    tf.keras.utils.to_categorical(y_test, num_classes=n_classes)
)

In [ ]:
history=model.fit(train_data_generator, validation_data=test_data_generator, epochs=3)

Epoch 1/3


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


92/92 ━━━━━━━━━━━━━━━━━━━━ 69s 545ms/step - accuracy: 0.2706 - loss: 1.5961 - val_accuracy: 0.3351 - val_loss: 1.5565
Epoch 2/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 21s 229ms/step - accuracy: 0.3521 - loss: 1.5345 - val_accuracy: 0.3460 - val_loss: 1.4638
Epoch 3/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 22s 237ms/step - accuracy: 0.3765 - loss: 1.4410 - val_accuracy: 0.3801 - val_loss: 1.3999


In [ ]:
base_model.trainable=True #enable fine-tuning

In [ ]:
model_fined = Sequential([
    base_model,
    Flatten(),
    Dense(128, activation='relu'),
    Dense(5, activation='softmax')
]
)

In [ ]:
model_fined.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

In [ ]:
history_fine=model_fined.fit(train_data_generator, validation_data=test_data_generator, epochs=3)

Epoch 1/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 28s 284ms/step - accuracy: 0.2860 - loss: 1.5951 - val_accuracy: 0.3392 - val_loss: 1.5474
Epoch 2/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 22s 237ms/step - accuracy: 0.3638 - loss: 1.5212 - val_accuracy: 0.3515 - val_loss: 1.4567
Epoch 3/3
92/92 ━━━━━━━━━━━━━━━━━━━━ 21s 228ms/step - accuracy: 0.4085 - loss: 1.4289 - val_accuracy: 0.3747 - val_loss: 1.3995
